# Démonstration cross-engine runtime EVPI/EVSI — tranche 3/3 (issue #13569)

Ce notebook de démonstration exécute le contrat `VoiContract` sur **les deux moteurs natifs du dépôt** (PyMC et Infer.NET) et compare leurs sorties à la **référence analytique close-form NumPy**.

## Acceptance

1. **Même problème envoyé aux deux moteurs** : on sérialise un `VoiContract` JSON et on le passe à `adapter_pymc.run_pymc` et `adapter_infernet.run_infernet`.
2. **Contrôles négatifs** : `EVSI=0` (test sans valeur), `EVSI=EVPI` (test parfait).
3. **Contrôle discriminant** : `0 < EVSI nette < EVPI`.
4. **Tableau d'accord/désaccord** : `compare.compare` produit un `CompareReport` sérialisable.

## Scope

`MyIA.AI.Notebooks/Probas/DecisionTheory/voi/` : `__init__.py`, `contract.py`, `adapter_pymc.py`, `adapter_infernet.py`, `compare.py`, `tests/`.

Référence canonique : `IIT/ICT-Series/ict/voi.py` (tranche 1/3, PR #13652).

## Limitations

- L'adaptateur **Infer.NET** nécessite `dotnet` + `dotnet-script` dans le PATH (RECOVERABLE-LOCAL — voir `sota-not-workaround.md` §F).
- L'adaptateur **PyMC** nécessite `pymc` installé (RECOVERABLE-LOCAL).
- Si l'un des deux adaptateurs échoue, le comparateur rapporte l'erreur dans `diffs` et continue — **pas de moyenne masquée**.

In [ ]:
import sys, os
_HERE = os.getcwd()
_ROOT = os.path.dirname(_HERE)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import numpy as np
from Probas.DecisionTheory.voi import contract as contract_mod
from Probas.DecisionTheory.voi import compare as compare_mod

## 1. Scénario parapluie (DecPyMC-5 section 2)

In [ ]:
parapluie = contract_mod.VoiContract(
    states=("pluie", "soleil"),
    prior=(0.3, 0.7),
    actions=("parapluie", "pas_parapluie"),
    utility=((0.0, -50.0), (-5.0, 0.0)),
    likelihood=((0.8, 0.2), (0.1, 0.9)),
    test_outcomes=("annonce_pluie", "annonce_soleil"),
    cost=1.0,
)
analytical = contract_mod.animat_decision_summary_contract(parapluie)
print(f"EU sans info = {analytical.eu_no_info:.4f} (best: {analytical.best_no_info})")
print(f"EVPI         = {analytical.evpi:.4f}")
print(f"EVSI brute   = {analytical.evsi:.4f}")
print(f"EVSI nette   = {analytical.evsi_net:.4f}")
print(f"Observer ?   = {analytical.observe}")

## 2. Comparaison cross-engine runtime

In [ ]:
report = compare_mod.compare(
    parapluie,
    include_pymc=True,
    include_infernet=True,
    tolerance=1e-1,
)
print(f"Accord analytique <-> PyMC        : {report.pymc is not None}")
print(f"Accord analytique <-> Infer.NET   : {report.infernet is not None}")
print(f"Agreement global                  : {report.agreement}")
print(f"Divergences rapportées            : {len(report.diffs)}")
for d in report.diffs:
    print(f"  - {d}")

## 3. Contrôle négatif : test qui ne change jamais la décision (EVSI=0)

In [ ]:
deg = contract_mod.VoiContract(
    states=("a", "b"),
    prior=(0.5, 0.5),
    actions=("parapluie", "pas_parapluie"),
    utility=((0.0, -50.0), (-5.0, 0.0)),
    # Test degenere : likelihood = prior (info independante de l'etat).
    likelihood=((0.5, 0.5), (0.5, 0.5)),
    test_outcomes=("o1", "o2"),
    cost=0.0,
)
deg_res = contract_mod.animat_decision_summary_contract(deg)
print(f"EVSI degenerated test = {deg_res.evsi:.6f}  (attendu : 0)")
assert abs(deg_res.evsi) < 1e-9
print("OK : test degenerated => EVSI=0, conforme a la borne inferieure.")

## 4. Sérialisation JSON pour runner multi-wakeup

In [ ]:
import json
report_dict = report.to_dict()
report_json = json.dumps(report_dict, indent=2)
print(f"Rapport serialisable : {len(report_json)} caracteres.")
print(f"Clefs du rapport : {sorted(report_dict.keys())}")